## Section 1: Import Required Libraries

# Telecom Network Telemetry - Exploratory Data Analysis (EDA)

## Objective
Analyze telemetry data from telecom infrastructure to:
1. Understand data structure and quality
2. Identify network health patterns and anomalies
3. Establish baseline metrics for predictive modeling
4. Define Key Performance Indicators (KPIs)
5. Discover failure precursors and risk indicators

**Output**: Foundation for AI-powered network diagnostics agent development

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import zscore, iqr
import warnings
warnings.filterwarnings('ignore')

# Configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

## Section 2: Load and Inspect Data

In [4]:
# Load telemetry data
df = pd.read_csv(r'D:\TELECOM_project\data\raw\telemetry_simulatory_dataset.csv', on_bad_lines='skip')

print("Dataset Loaded Successfully!")
print(f"Dataset Shape: {df.shape}\n")
df.head()

Dataset Loaded Successfully!
Dataset Shape: (790, 9)



,timestamp,device_id,cpu_percent,memory_percent,latency,bandwidth_out,bandwidth_in,packet_loss,crc_errors
0,2026-03-15 17:53:42.265654,Router 1,24.2,87.3,23.844750,0.000000,0.000000,0.459339,1
1,2026-03-15 17:53:43.290376,Router 1,32.2,87.4,25.051141,0.000201,0.000535,0.354818,1
2,2026-03-15 17:53:44.318805,Router 1,14.3,87.4,23.236819,0.001492,0.000878,0.308739,0
3,2026-03-15 17:53:45.345802,Router 1,13.3,87.4,11.654113,0.007608,0.005952,0.452141,1
4,2026-03-15 17:53:46.379155,Router 1,11.8,87.4,13.208884,0.005281,0.000124,0.341741,2


In [5]:
df=df.drop(columns=['device_id'])
df.head()

,timestamp,cpu_percent,memory_percent,latency,bandwidth_out,bandwidth_in,packet_loss,crc_errors
0,2026-03-15 17:53:42.265654,24.2,87.3,23.844750,0.000000,0.000000,0.459339,1
1,2026-03-15 17:53:43.290376,32.2,87.4,25.051141,0.000201,0.000535,0.354818,1
2,2026-03-15 17:53:44.318805,14.3,87.4,23.236819,0.001492,0.000878,0.308739,0
3,2026-03-15 17:53:45.345802,13.3,87.4,11.654113,0.007608,0.005952,0.452141,1
4,2026-03-15 17:53:46.379155,11.8,87.4,13.208884,0.005281,0.000124,0.341741,2


In [6]:
features=df[["cpu_percent",	"memory_percent","latency","bandwidth_out",	"bandwidth_in",	"packet_loss","crc_errors"]]

In [7]:
from sklearn.ensemble import IsolationForest

model=IsolationForest(contamination=0.05,random_state=42)

model.fit(features)

df["anomaly"]=model.predict(features)

In [8]:
df.head()

,timestamp,cpu_percent,memory_percent,latency,bandwidth_out,bandwidth_in,packet_loss,crc_errors,anomaly
0,2026-03-15 17:53:42.265654,24.2,87.3,23.844750,0.000000,0.000000,0.459339,1,1
1,2026-03-15 17:53:43.290376,32.2,87.4,25.051141,0.000201,0.000535,0.354818,1,1
2,2026-03-15 17:53:44.318805,14.3,87.4,23.236819,0.001492,0.000878,0.308739,0,1
3,2026-03-15 17:53:45.345802,13.3,87.4,11.654113,0.007608,0.005952,0.452141,1,1
4,2026-03-15 17:53:46.379155,11.8,87.4,13.208884,0.005281,0.000124,0.341741,2,1


In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

model.fit(features_scaled)
df["anomaly"] = model.predict(features_scaled)

In [10]:
df["anomaly_label"] = df["anomaly"].map({1: "Normal", -1: "Anomaly"})

In [11]:
df.head()

,timestamp,cpu_percent,memory_percent,latency,bandwidth_out,bandwidth_in,packet_loss,crc_errors,anomaly,anomaly_label
0,2026-03-15 17:53:42.265654,24.2,87.3,23.844750,0.000000,0.000000,0.459339,1,1,Normal
1,2026-03-15 17:53:43.290376,32.2,87.4,25.051141,0.000201,0.000535,0.354818,1,1,Normal
2,2026-03-15 17:53:44.318805,14.3,87.4,23.236819,0.001492,0.000878,0.308739,0,1,Normal
3,2026-03-15 17:53:45.345802,13.3,87.4,11.654113,0.007608,0.005952,0.452141,1,1,Normal
4,2026-03-15 17:53:46.379155,11.8,87.4,13.208884,0.005281,0.000124,0.341741,2,1,Normal


In [12]:
pd.DataFrame(df).to_csv(r"D:\\TELECOM_project\\data\\processed\\telemetry_simulatory_dataset.csv", header=True, index=False)

In [13]:
print(df["anomaly"].value_counts())

anomaly
 1    750
-1     40
Name: count, dtype: int64


In [14]:
#incepting anomally 

anomallies=df[df["anomaly"]==-1]
anomallies.head(10)

,timestamp,cpu_percent,memory_percent,latency,bandwidth_out,bandwidth_in,packet_loss,crc_errors,anomaly,anomaly_label
36,2026-03-15 17:55:23.065685,19.8,88.0,45.256776,0.074528,0.002704,0.262877,1,-1,Anomaly
86,2026-03-15 17:56:14.547243,25.3,86.8,31.371873,0.001117,0.049843,0.172329,0,-1,Anomaly
176,2026-03-15 17:57:47.249023,31.1,88.2,28.020916,0.047649,0.066416,0.428490,0,-1,Anomaly
184,2026-03-15 17:57:55.475346,22.5,89.1,33.661431,0.004595,0.081964,0.204251,1,-1,Anomaly
185,2026-03-15 17:57:56.507672,37.8,90.0,13.490948,0.058745,0.096199,0.223715,0,-1,Anomaly
186,2026-03-15 17:57:57.535649,32.2,89.8,28.739306,0.002557,0.052716,0.477191,0,-1,Anomaly
187,2026-03-15 17:57:58.562134,30.5,90.0,23.952228,0.015973,0.074993,0.503200,0,-1,Anomaly
191,2026-03-15 17:58:02.674737,26.5,90.3,26.672356,0.057094,0.012351,0.452507,2,-1,Anomaly
201,2026-03-15 17:58:12.980693,15.7,88.7,12.475811,0.009897,0.014838,0.082726,0,-1,Anomaly
251,2026-03-15 17:59:04.467401,34.8,84.2,26.135226,0.009562,0.007240,0.440356,0,-1,Anomaly


In [15]:
pd.DataFrame(anomallies).to_csv(r"D:\\TELECOM_project\\data\\anomalies\\anomalies_only.csv",header=True,index=False)
